# Improved LLM Plans
Original feature, Only one mental Keggle Mental health information used.
Then, based on information, how should I imporve my model,
In inital Training, gpt model4, and we have serveral information inside.
Next approch, I like to Hugging Datasets and Mental health datasets

Currently, feature we just populate randomly.
Next what, I want is text related llm topic clustering,
Based on token, clustering.


In [1]:
# Hugging Datasets called
raw_data_path = "raw_data"
cleaned_data_path= "cleaned_data"



In [2]:
import pandas as pd
# Inital Raw Data Used.

def clean_keggle_df(data_path):
    df = pd.read_csv(data_path)
    df = df[["questionText", "topics", "re_diagnosis","clean_answer_text"]]
    # Lower case
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    # remove non-world
    df = df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    # remove number
    df = df.replace(to_replace=r'\d', value='', regex=True)

    return df

def clean_hugging_df(data_path):
    df = pd.read_csv(data_path)

    df = df[["questionTitle", "questionText", "topic", "answerText"]]
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    df["questionText"] = df["questionTitle"].fillna('') + " " + df["questionText"].fillna('')
    df = df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    df = df.replace(to_replace=r'\d', value='', regex=True)
    df = df[["questionText", "topic", "answerText"]]

    # print(hugging_df.head)
    return df

keggle_df = clean_keggle_df(f"{raw_data_path}/counsel_cleaned.csv")   
print(keggle_df.shape)
keggle_df.to_csv(f"{cleaned_data_path}/cleaned_counsel.csv")
hugging_df = clean_hugging_df(f"{raw_data_path}/huggin_counsel_chat.csv")
print(hugging_df.shape)
hugging_df

C:\Users\ykim\AppData\Local\Temp\ipykernel_18888\1977113534.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
C:\Users\ykim\AppData\Local\Temp\ipykernel_18888\1977113534.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


(1373, 4)
(2775, 3)


,questionText,topic,answerText
0,do i have too many issues for counseling i hav...,depression,it is very common for people to have multiple ...
1,do i have too many issues for counseling i hav...,depression,ive never heard of someone having too many iss...
2,do i have too many issues for counseling i hav...,depression,absolutely not i strongly recommending workin...
3,do i have too many issues for counseling i hav...,depression,let me start by saying there are never too man...
4,do i have too many issues for counseling i hav...,depression,i just want to acknowledge you for the courage...
...,...,...,...
2770,are some clients more difficult than others wh...,counselingfundamentals,although many clients have the capacity to be ...
2771,are some clients more difficult than others wh...,counselingfundamentals,i usually dont label a client as difficult bec...
2772,are some clients more difficult than others wh...,counselingfundamentals,dang right heh heh and correct me if im wrong...
2773,are some clients more difficult than others wh...,counselingfundamentals,yes just like some relationships outside of ou...


In [3]:
import pandas as pd

# Read File Information
hugging_df = pd.read_csv(f"{cleaned_data_path}/cleaned_hugging.csv")
counsel_df = pd.read_csv(f"{cleaned_data_path}/cleaned_counsel.csv")

# Change Column Name
hugging_df.rename(columns={"questionText": "question_text",
                           "topic": "topics", 
                           "answerText": "answer_text"}, inplace=True)
hugging_df.drop(columns=['Unnamed: 0'], inplace=True)

print("After renaming:", hugging_df.columns)
# Drop unused column
counsel_df.drop(columns=['Unnamed: 0', 're_diagnosis'], inplace=True)
print(counsel_df.columns)
counsel_df.rename(columns={"questionText": "question_text", 
                           "clean_answer_text": "answer_text"}, inplace=True)

# Select target column
hugging_df = hugging_df[["question_text", "topics", "answer_text"]]
counsel_df = counsel_df[["question_text", "topics", "answer_text"]]
print(hugging_df.columns)
print(counsel_df.columns)

# Concat Column
combined_dataset = pd.concat([hugging_df, counsel_df], ignore_index=True)

# Combined Output
print(combined_dataset.columns)
combined_dataset.to_csv(f"{cleaned_data_path}/combined_output.csv")


After renaming: Index(['questionTitle', 'question_text', 'topics', 'answer_text'], dtype='object')
Index(['questionText', 'topics', 'clean_answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')


In [4]:
# Chat Promt, Design.
combined_dataset = pd.read_csv(f"{cleaned_data_path}/combined_output.csv")
print(combined_dataset.columns)



Index(['Unnamed: 0', 'question_text', 'topics', 'answer_text'], dtype='object')


In [10]:
# NLTK test
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
text = "This is an example. Here is another sentence."
sentences = sent_tokenize(text)

print(sentences)


['This is an example.', 'Here is another sentence.']


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ykim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [11]:
# Cleaned Combined Datasets too shorts and too long
import re
from nltk.tokenize import word_tokenize

def is_noisy(text: str) -> bool:
    if re.search(r'[가-힣A-Za-z]', text) is None:
        return True
    cleaned = re.sub(r'[^\w\s]', '', text) 
    if len(cleaned) == 0 or len(cleaned) < len(text) * 0.02:  
        return True
    return False

def clean_combined_dataset(df):
    def token_count(text):
        tokens = sent_tokenize(text)
        return len(tokens)
    # Apply the token_count function to calculate the number of tokens in questions and answers
    df = df.dropna()
    df['q_token_count'] = df['question_text'].apply(token_count)
    df['a_token_count'] = df['answer_text'].apply(token_count)
    
    return df
# print(combined_dataset.columns)
print(f"Before: {combined_dataset.shape}")
cleaned_combined_df = clean_combined_dataset(combined_dataset)
cleaned_combined_df.to_csv(f"{cleaned_data_path}/cleaned_combined.csv")
print(f"After: {cleaned_combined_df.shape}")


Before: (4148, 4)
After: (3985, 6)


C:\Users\ykim\AppData\Local\Temp\ipykernel_18888\339852553.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['q_token_count'] = df['question_text'].apply(token_count)
C:\Users\ykim\AppData\Local\Temp\ipykernel_18888\339852553.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['a_token_count'] = df['answer_text'].apply(token_count)


In [21]:
# Lamma testing Device
from datasets import Dataset, DatasetDict


target_df = pd.read_csv(f"{cleaned_data_path}/cleaned_combined.csv")
print(f"After: {target_df.shape}")
print(f"Columns: {target_df.columns}")

system_prompt = (
    "This is a Mental Health ChatBot assistant designed based on actual consultation data and implemented using real test cases." 
    "Its purpose is to accurately understand users' questions and respond with comforting and helpful messages."
    "The focus is primarily on the question_text, and when necessary, it can provide various empathetic expressions. "
    "In cases where the user's question is unclear or emotionally unstable, the assistant should request additional clarification, while also offering basic empathy and supportive advice"
)

def format_to_chat_messages(example):
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": example["question_text"] + f" based on {example['topics']}"},
            {"role": "assistant", "content": example["answer_text"]},
        ]
    }

# Grab only nessary columns only
target_df = target_df[["question_text", "topics", "answer_text"]].dropna()
hf_dataset = Dataset.from_pandas(target_df)

# Hugging Face Datasets
hf_dataset = hf_dataset.map(format_to_chat_messages)
hf_dataset = hf_dataset.remove_columns([col for col in hf_dataset.column_names if col != "messages"])
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)

train_data_path = "train_data"
hf_dataset["train"].to_json(f"{train_data_path}/train_dataset.json", orient="records", force_ascii=False)
hf_dataset["test"].to_json(f"{train_data_path}/test_dataset.json", orient="records", force_ascii=False)
print(hf_dataset["train"][0])
print(hf_dataset["test"][0])



After: (3985, 7)
Columns: Index(['Unnamed: 0.1', 'Unnamed: 0', 'question_text', 'topics', 'answer_text',
       'q_token_count', 'a_token_count'],
      dtype='object')


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 87.99ba/s]

{'messages': [{'content': "This is a Mental Health ChatBot assistant designed based on actual consultation data and implemented using real test cases.Its purpose is to accurately understand users' questions and respond with comforting and helpful messages.The focus is primarily on the question_text, and when necessary, it can provide various empathetic expressions. In cases where the user's question is unclear or emotionally unstable, the assistant should request additional clarification, while also offering basic empathy and supportive advice", 'role': 'system'}, {'content': 'ive been experiencing a lot of anxiety and panic attacks lately i was recently diagnosed by my psychiatrist with obsessivecompulsive disorder lately ive been questioning everything from my career to my relationship my boyfriend and i just moved in a few months ago all of a sudden i dont feel as comfortable around him as i used to although i cant seem to find a reason as to why i feel this way based on anxiety', '

In [30]:
import os
import random
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, set_seed
from trl import SFTTrainer

# === 사용자 설정 ===
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"  # or Falcon-RW-1B
DATA_DIR = "./train_data"
OUTPUT_DIR = "./llama3-mental-health"
MAX_SEQ_LENGTH = 512

# === Chat Template (LLaMA3 스타일) ===
LLAMA_3_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}{{ message['content'] }}"
    "{% elif message['role'] == 'user' %}{{ '\n\nHuman: ' + message['content'] + eos_token }}"
    "{% elif message['role'] == 'assistant' %}{{ '\n\nAssistant: ' + message['content'] + eos_token }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '\n\nAssistant: ' }}{% endif %}"
)

# === Step 1: 데이터 로드 ===
train_df = pd.read_json(f"{DATA_DIR}/train_dataset.json", lines = True)
test_df = pd.read_json(f"{DATA_DIR}/test_dataset.json", lines = True)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# === Step 2: 토크나이저 설정 ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = LLAMA_3_CHAT_TEMPLATE.replace("{{ eos_token }}", tokenizer.eos_token)
tokenizer.padding_side = "right"

# === Step 3: 메시지 → 토큰화 ===
def convert(example):
    prompt = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return tokenizer(prompt, truncation=True, padding="max_length", max_length=MAX_SEQ_LENGTH)

train_dataset = train_dataset.map(convert, remove_columns=["messages"])
test_dataset = test_dataset.map(convert, remove_columns=["messages"])

# === Step 4: 모델 로드 ===
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# === Step 5: 학습 설정 ===
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    save_strategy="epoch",
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none",
)

# === Step 6: SFTTrainer 설정 ===
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=training_args,
    dataset_text_field="input_ids",
    packing=False
)

# === Step 7: 학습 실행 ===
trainer.train()

# === Step 8: 저장 ===
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 모델 저장 완료: {OUTPUT_DIR}")


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
401 Client Error. (Request ID: Root=1-68068d5d-594ff449399ed85d3d330bbf;e3795831-21c0-4f6d-9574-2de08e6fd0da)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.